# Домашнее задание 1

Реализуйте три алгоритма:
1. **OVA (One-vs-All)** - мультиклассовая классификация через K бинарных классификаторов
2. **AVA (All-vs-All)** - мультиклассовая классификация через K(K-1)/2 бинарных классификаторов
3. **LARS (Least Angle Regression)** - алгоритм построения пути регрессионных коэффициентов

Проверка: `uv run python -m unittest tests/*.py`

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from copy import deepcopy
from itertools import combinations
from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification, make_regression
from sklearn.metrics import accuracy_score
from matplotlib.colors import ListedColormap

## 1. One-vs-All (OVA)

Стратегия OVA (она же One-vs-Rest, OVR):
- Для задачи с K классами обучаем K бинарных классификаторов
- Классификатор $k$ обучается отличать класс $k$ от всех остальных
- При предсказании выбираем класс с наибольшей уверенностью (confidence)

Реализуйте класс `OVAClassifier`. В качестве базового классификатора используйте sklearn `LogisticRegression`.

In [ ]:
class OVAClassifier:
    def __init__(self, base_clf=None):
        """
        Parameters
        ----------
        base_clf : sklearn-compatible binary classifier
            If None, uses LogisticRegression(max_iter=1000).
            Use deepcopy(base_clf) when creating copies for each class.
        """
        self.base_clf = base_clf if base_clf is not None else LogisticRegression(max_iter=1000)
        # TODO: initialize any needed attributes

    def fit(self, X, y):
        """
        Train K binary classifiers, one per class.
        For class k: y_binary = (y == k).astype(int)

        Store:
        - self.classes_ : np.array of unique class labels
        - self.classifiers_ : list of fitted binary classifiers
        """
        # TODO: implement
        pass

    def predict(self, X):
        """
        For each sample, return the class whose classifier gives
        the highest confidence. Use decision_function if available,
        otherwise predict_proba[:, 1].

        Returns
        -------
        np.array of shape (n_samples,)
        """
        # TODO: implement
        pass

### Проверка OVA

In [ ]:
# Генерируем 4-класcовые 2D данные для визуализации
if __name__ != "homework":
    X_vis, y_vis = make_classification(
        n_samples=300, n_features=2, n_informative=2, n_redundant=0,
        n_classes=4, n_clusters_per_class=1, random_state=42,
    )
    X_vis_train, X_vis_test = X_vis[:200], X_vis[200:]
    y_vis_train, y_vis_test = y_vis[:200], y_vis[200:]

In [ ]:
def plot_decision_boundary(clf, X, y, ax, title):
    """Plot decision regions for a multiclass classifier on 2D data."""
    eps = 0.5
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - eps, X[:, 0].max() + eps, 300),
        np.linspace(X[:, 1].min() - eps, X[:, 1].max() + eps, 300),
    )
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    cmap_bg = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF', '#FFFFAA'])
    ax.pcolormesh(xx, yy, Z, cmap=cmap_bg, alpha=0.4)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='tab10', edgecolors='k', s=30)
    ax.set_title(title)
    ax.grid(alpha=0.2)

# Helper is always defined, but check cells below only run interactively

In [ ]:
if __name__ != "homework":
    from sklearn.multiclass import OneVsRestClassifier

    ova = OVAClassifier(LogisticRegression(max_iter=1000))
    ova.fit(X_vis_train, y_vis_train)
    ova_preds = ova.predict(X_vis_test)

    sk_ova = OneVsRestClassifier(LogisticRegression(max_iter=1000))
    sk_ova.fit(X_vis_train, y_vis_train)
    sk_ova_preds = sk_ova.predict(X_vis_test)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    plot_decision_boundary(ova, X_vis_test, y_vis_test, axes[0], 'Our OVA')
    plot_decision_boundary(sk_ova, X_vis_test, y_vis_test, axes[1], 'sklearn OneVsRest')
    plt.tight_layout()
    plt.show()

    print(f'Our OVA accuracy:     {accuracy_score(y_vis_test, ova_preds):.3f}')
    print(f'sklearn OVR accuracy: {accuracy_score(y_vis_test, sk_ova_preds):.3f}')

## 2. All-vs-All (AVA)

Стратегия AVA (она же One-vs-One, OVO):
- Для задачи с K классами обучаем $\frac{K(K-1)}{2}$ бинарных классификаторов
- Классификатор $(i, j)$ обучается только на объектах классов $i$ и $j$
- При предсказании каждый классификатор голосует за один из своих двух классов
- Побеждает класс с наибольшим числом голосов

AVA обучает больше классификаторов, но каждый на меньшем подмножестве данных.

In [ ]:
class AVAClassifier:
    def __init__(self, base_clf=None):
        """
        Parameters
        ----------
        base_clf : sklearn-compatible binary classifier
            If None, uses LogisticRegression(max_iter=1000).
        """
        self.base_clf = base_clf if base_clf is not None else LogisticRegression(max_iter=1000)
        # TODO: initialize any needed attributes

    def fit(self, X, y):
        """
        Train K*(K-1)/2 binary classifiers, one per pair of classes.
        For pair (i, j): use only samples where y is i or j.

        Store:
        - self.classes_ : np.array of unique class labels
        - self.classifiers_ : list of (clf, class_i, class_j) tuples
        """
        # TODO: implement
        pass

    def predict(self, X):
        """
        For each sample, each classifier votes for one of its two classes.
        Return the class with the most votes.
        Ties can be broken arbitrarily.

        Returns
        -------
        np.array of shape (n_samples,)
        """
        # TODO: implement
        pass

### Проверка AVA

In [ ]:
if __name__ != "homework":
    from sklearn.multiclass import OneVsOneClassifier

    ava = AVAClassifier(LogisticRegression(max_iter=1000))
    ava.fit(X_vis_train, y_vis_train)
    ava_preds = ava.predict(X_vis_test)

    sk_ava = OneVsOneClassifier(LogisticRegression(max_iter=1000))
    sk_ava.fit(X_vis_train, y_vis_train)
    sk_ava_preds = sk_ava.predict(X_vis_test)

    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    plot_decision_boundary(ova, X_vis_test, y_vis_test, axes[0], 'Our OVA')
    plot_decision_boundary(ava, X_vis_test, y_vis_test, axes[1], 'Our AVA')
    plot_decision_boundary(sk_ava, X_vis_test, y_vis_test, axes[2], 'sklearn OneVsOne')
    plt.tight_layout()
    plt.show()

    print(f'Our OVA accuracy:     {accuracy_score(y_vis_test, ova_preds):.3f}')
    print(f'Our AVA accuracy:     {accuracy_score(y_vis_test, ava_preds):.3f}')
    print(f'sklearn OVO accuracy: {accuracy_score(y_vis_test, sk_ava_preds):.3f}')

## 3. LARS (Least Angle Regression)

LARS - алгоритм, который строит путь регрессионных коэффициентов от нуля до полного OLS-решения. На каждом шаге он добавляет в модель один признак.

Алгоритм:
1. Начинаем с $w = 0$, остаток $r = y$
2. Находим признак $x_j$, наиболее коррелированный с остатком $r$
3. Двигаем коэффициент $w_j$ в направлении корреляции, пока другой признак $x_k$ не станет столь же коррелированным с остатком
4. Двигаем оба коэффициента в equiangular direction (направлении равных углов)
5. Повторяем, пока не будут задействованы все признаки или не достигнут лимит `n_features`

**Важно:** данные должны быть стандартизированы (mean=0) перед подачей в LARS.

In [ ]:
class LARS:
    def __init__(self, n_features=None):
        """
        Parameters
        ----------
        n_features : int or None
            Maximum number of features to select.
            If None, use all features.
        """
        self.n_features = n_features

    def fit(self, X, y):
        """
        Fit LARS on standardized data (X and y should be centered).

        Algorithm sketch:
        - mu = 0 (current prediction), residual = y - mu
        - Repeat:
            1. c = X^T @ residual  (correlations)
            2. C = max(|c|)        (max absolute correlation)
            3. Active set A = {j : |c_j| = C}
            4. Compute equiangular direction:
               s_A = sign(c_A)
               X_A = X[:, A] * s_A
               G_A = X_A^T @ X_A
               A_A = (1^T G_A^{-1} 1)^{-1/2}
               u_A = X_A @ (A_A * G_A^{-1} @ 1)
            5. Compute step size gamma:
               a = X^T @ u_A
               gamma = min_{j not in A}+( (C - c_j)/(A_A - a_j), (C + c_j)/(A_A + a_j) )
            6. mu += gamma * u_A
            7. Update coefficients

        Store:
        - self.coef_ : np.array of shape (n_features_total,)
        - self.intercept_ : float (0.0 for centered data)
        - self.coef_path_ : list of np.arrays, coefficients at each step
        """
        # TODO: implement
        pass

    def predict(self, X):
        """
        Predict: X @ self.coef_ + self.intercept_

        Returns
        -------
        np.array of shape (n_samples,)
        """
        # TODO: implement
        pass

### Проверка LARS

In [ ]:
if __name__ != "homework":
    # Sparse regression: 10 features, only 3 informative
    X_lars, y_lars, true_coef = make_regression(
        n_samples=200, n_features=10, n_informative=3,
        noise=5.0, coef=True, random_state=42,
    )
    # Standardize
    X_lars = (X_lars - X_lars.mean(axis=0)) / X_lars.std(axis=0)
    y_lars = y_lars - y_lars.mean()

In [ ]:
if __name__ != "homework":
    from sklearn.linear_model import Lars as SkLars

    lars = LARS()
    lars.fit(X_lars, y_lars)

    sk_lars = SkLars().fit(X_lars, y_lars)

In [ ]:
if __name__ != "homework":
    # Coefficient path: features entering the model one by one
    path = np.array(lars.coef_path_)

    plt.figure(figsize=(12, 6))
    for j in range(path.shape[1]):
        plt.plot(path[:, j], label=f'feature {j}')
    plt.xlabel('LARS step')
    plt.ylabel('coefficient value')
    plt.title('LARS coefficient path')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

In [ ]:
if __name__ != "homework":
    # Compare final coefficients: ours vs sklearn vs true
    feat_idx = np.arange(X_lars.shape[1])
    width = 0.25

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(feat_idx - width, true_coef, width, label='True coefficients', alpha=0.8)
    ax.bar(feat_idx, lars.coef_, width, label='Our LARS', alpha=0.8)
    ax.bar(feat_idx + width, sk_lars.coef_, width, label='sklearn Lars', alpha=0.8)
    ax.set_xlabel('Feature index')
    ax.set_ylabel('Coefficient value')
    ax.set_title('Coefficient comparison')
    ax.set_xticks(feat_idx)
    ax.legend()
    ax.grid(alpha=0.2, axis='y')
    plt.tight_layout()
    plt.show()

    our_mse = np.mean((lars.predict(X_lars) - y_lars) ** 2)
    sk_mse = np.mean((sk_lars.predict(X_lars) - y_lars) ** 2)
    print(f'Our LARS MSE:    {our_mse:.4f}')
    print(f'sklearn Lars MSE: {sk_mse:.4f}')